# AMEX Enterprise Credit Risk Platform
## Notebook 36 -- Early Payment Default: Validation & Deployment
### Phase 2 . Problem Statement 5: Early Payment Default Detection

CRISP-DM stage: **Evaluation & Deployment**. Sprint 1, Notebook 3 of 4 for this problem. Depends on Problem 1 Notebooks 01-05 and this problem's own Notebooks 34-35 (reads `notebook_35_summary.json` -> `early_window_modeling_results.json` for the real, measured per-K AUC-retention results) -- no dependency on Problem 3 or Problem 4.

**What this notebook does (real, computed on your machine when you run it):**
- Selects the window to validate and deploy: the earliest candidate K that met Notebook 34's AUC-retention KPI, or -- if none did -- the best-performing candidate, carried through and clearly flagged **NOT RECOMMENDED FOR PRODUCTION** rather than hidden
- Rebuilds that K's restricted feature set and retrains the final model, deterministically reproducing Notebook 35's reported metrics (checked, not assumed)
- Runs real statistical validation: a 2,000-resample bootstrap confidence interval on holdout AUC, a real calibration check (predicted-PD deciles vs. observed default rate), and a split-half population-stability (PSI) check on the predicted score
- States an explicit, honest deployment-scope limitation (this model assumes exactly this many early statements are available -- more history should go to Problem 1's full-history champion instead)
- Persists the model + preprocessing artifacts, generates a real, runnable FastAPI scoring service (`early_default_service.py`), and proves it live via a self-test that imports the exact file just written to disk and checks its output against a direct computation
- Produces a Word report (`Early_Default_Validation_Deployment_Report.docx`) combining the validation summary, honest limitation, deployment readiness checklist, and charts

**What this notebook does NOT do:** financial-impact reporting and final repository packaging -- that's Notebook 37.

Zero-fabrication: every metric in this notebook is computed live from your real Kaggle data and Notebook 35's real results on this run -- including the possibility, reported plainly if it happens, that no candidate window met the KPI target.


In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOKS 01-05, 34, 35
# =============================================================================
import os
import sys
import gc
import json
import time
import warnings
import importlib.util
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01-05, 34, 35")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB34_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_34_summary.json"
NB35_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_35_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
    (NB34_SUMMARY_PATH, "run 34_early_payment_default_business_understanding.ipynb first"),
    (NB35_SUMMARY_PATH, "run 35_early_payment_default_modeling.ipynb first -- this notebook "
                         "consumes its real per-K AUC-retention results, not a guess"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB35_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB35_SUMMARY = json.load(f)

MODELING_RESULTS_PATH = Path(NB35_SUMMARY["modeling_results_path"])
if not MODELING_RESULTS_PATH.exists():
    raise FileNotFoundError(f"{MODELING_RESULTS_PATH} not found.\nFix: re-run 35_early_payment_default_modeling.ipynb.")
with open(MODELING_RESULTS_PATH, "r", encoding="utf-8") as f:
    MODELING_ARTIFACT = json.load(f)

RESULTS_BY_K = {int(k): v for k, v in MODELING_ARTIFACT["results_by_k"].items()}
KS_MEETING_KPI = MODELING_ARTIFACT["ks_meeting_kpi_target"]
KPI_TARGETS = MODELING_ARTIFACT["kpi_targets"]
FULL_HISTORY_AUC = MODELING_ARTIFACT["full_history_reference_auc"]

# --- Pick the window this notebook validates and deploys. If one or more
#     candidates met Notebook 34's KPI target, the earliest (shortest,
#     operationally best) one wins -- shortest early-observation window is
#     more valuable for early intervention, all else equal. If NONE met the
#     KPI, this is reported plainly (not hidden): the best-AUC candidate is
#     still carried through validation and deployment-packaging for
#     completeness, but flagged NOT RECOMMENDED FOR PRODUCTION throughout --
#     an honest "not yet viable" outcome, same standard Problem 2's
#     fair-lending section holds itself to, rather than silently deploying
#     something that didn't clear its own bar. ---
if KS_MEETING_KPI:
    WINNING_K = min(KS_MEETING_KPI)
    MEETS_KPI = True
    print(f"Candidates meeting the {KPI_TARGETS['min_auc_retention_vs_full_history']:.0%} AUC-retention KPI: {KS_MEETING_KPI}")
    print(f"Selected WINNING_K = {WINNING_K} (earliest/shortest window that still clears the KPI)")
else:
    WINNING_K = max(RESULTS_BY_K.keys(), key=lambda k: RESULTS_BY_K[k]["holdout_auc"])
    MEETS_KPI = False
    print(
        f"HONEST FINDING (carried forward from Notebook 35): none of the candidate windows "
        f"{sorted(RESULTS_BY_K.keys())} met the {KPI_TARGETS['min_auc_retention_vs_full_history']:.0%} "
        f"AUC-retention KPI on the real run. Selected WINNING_K = {WINNING_K} (best holdout AUC among "
        f"candidates, {RESULTS_BY_K[WINNING_K]['holdout_auc']:.4f}) so this notebook can still validate and "
        f"package it -- but every section below marks it NOT RECOMMENDED FOR PRODUCTION. This is a real "
        f"outcome of this run's real data, not a defect in this notebook."
    )

WINNING_K_RESULT = RESULTS_BY_K[WINNING_K]
print(f"\nWINNING_K = {WINNING_K}")
print(json.dumps(WINNING_K_RESULT, indent=2))

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)
MAX_RAM_BYTES = _resource_limits.get("max_ram_bytes")

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_METRICS = NB05_SUMMARY["champion_metrics"]

if "early_payment_default_deployment" in PILLAR_DIRS:
    EPD_DEPLOYMENT_DIR = PILLAR_DIRS["early_payment_default_deployment"]
else:
    EPD_DEPLOYMENT_DIR = (
        PROJECT_ROOT / "Phase2_Regulatory_Loss_Provisioning"
        / "05_Problem5_Early_Payment_Default_Detection" / "deployment"
    )
    print(f"NOTE: 'early_payment_default_deployment' not in pillar_dirs -- using fallback: {EPD_DEPLOYMENT_DIR}")
EPD_DEPLOYMENT_DIR.mkdir(parents=True, exist_ok=True)
API_SUBDIR = EPD_DEPLOYMENT_DIR / "api"
API_SUBDIR.mkdir(parents=True, exist_ok=True)
MODELS_SUBDIR = EPD_DEPLOYMENT_DIR / "models"
MODELS_SUBDIR.mkdir(parents=True, exist_ok=True)

print(f"\nDeployment artifacts will be written under: {EPD_DEPLOYMENT_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import joblib
except ImportError:
    missing.append("joblib")
try:
    from sklearn.metrics import roc_auc_score
except ImportError:
    missing.append("scikit-learn")
try:
    from xgboost import XGBClassifier
except ImportError:
    missing.append("xgboost")
try:
    from fastapi.testclient import TestClient
except ImportError:
    missing.append("fastapi")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    from docx import Document
    from docx.shared import Inches
    from docx.enum.text import WD_ALIGN_PARAGRAPH
except ImportError:
    missing.append("python-docx")
try:
    import importlib.metadata as importlib_metadata
except ImportError:
    import importlib_metadata

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


def amex_metric_numpy(y_true: "np.ndarray", y_pred: "np.ndarray") -> float:
    """Official AMEX competition metric -- same implementation as Notebook 05
    Section 3 / Notebook 35 Section 2, reused here for consistency."""
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    def top_four_percent_captured(yt, yp):
        order = np.argsort(-yp, kind="mergesort")
        yt_sorted = yt[order]
        weight = np.where(yt_sorted == 0, 20.0, 1.0)
        cum_weight = np.cumsum(weight)
        cutoff = 0.04 * weight.sum()
        mask = cum_weight <= cutoff
        total_pos = yt_sorted.sum()
        if total_pos == 0:
            return 0.0
        return float(yt_sorted[mask].sum() / total_pos)

    def weighted_gini(yt, yp):
        order = np.argsort(-yp, kind="mergesort")
        yt_sorted = yt[order]
        weight = np.where(yt_sorted == 0, 20.0, 1.0)
        random_cum = np.cumsum(weight / weight.sum())
        total_pos_weighted = (yt_sorted * weight).sum()
        if total_pos_weighted == 0:
            return 0.0
        cum_pos_found = np.cumsum(yt_sorted * weight)
        lorentz = cum_pos_found / total_pos_weighted
        return float(((lorentz - random_cum) * weight).sum())

    g_actual = weighted_gini(y_true, y_pred)
    g_perfect = weighted_gini(y_true, y_true)
    normalized_gini = g_actual / g_perfect if g_perfect != 0 else 0.0
    top4 = top_four_percent_captured(y_true, y_pred)
    return 0.5 * (normalized_gini + top4)


logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (95% cap, WARP 6.4)")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
if MAX_RAM_BYTES:
    print(f"Configured RAM ceiling (90% of detected total): {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

# Same generic resolver Notebook 35 established, after finding
# project_config.json's own pillar_dirs entries for pre-reorg pillars
# (data_engineering, data_validation) still held stale, pre-reorg paths.
def _resolve_pillar_file(filename: str, pillar_key: str, legacy_folder_name: str,
                          stored_path_str: str = None, min_size: int = 10_000) -> Path:
    _candidates = [
        PROJECT_ROOT / "Phase1_Foundation" / "01_Problem1_Credit_Scoring_PD_Prediction"
        / legacy_folder_name / filename,
    ]
    if pillar_key in PILLAR_DIRS:
        _candidates.append(PILLAR_DIRS[pillar_key] / filename)
    _candidates.append(PROJECT_ROOT / legacy_folder_name / filename)
    if stored_path_str:
        _candidates.append(Path(stored_path_str))
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > min_size:
            return _c
    raise FileNotFoundError(
        f"Could not resolve a real, non-trivial {filename}. Checked:\n"
        + "\n".join(f"  - {c}" for c in _candidates)
        + "\nFix: run the notebook that produces this file again, or tell me the real path."
    )


_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )
RAW_TRAIN_LABELS_PATH = RAW_TRAIN_DATA_PATH.parent / "train_labels.csv"
if not RAW_TRAIN_LABELS_PATH.exists():
    raise FileNotFoundError(f"{RAW_TRAIN_LABELS_PATH} not found.")

TRAIN_SPLIT_PATH = _resolve_pillar_file(
    "train_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("train_split.csv"),
)
TEST_SPLIT_PATH = _resolve_pillar_file(
    "test_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("test_split.csv"),
)
VALIDATION_REPORT_PATH = _resolve_pillar_file(
    "data_validation_report.json", "data_validation", "Data_Validation", min_size=100,
)
with open(VALIDATION_REPORT_PATH, "r", encoding="utf-8") as f:
    VALIDATION_REPORT = json.load(f)

print(f"Raw train_data.csv  : {RAW_TRAIN_DATA_PATH}")
print(f"train_split.csv     : {TRAIN_SPLIT_PATH}")
print(f"test_split.csv      : {TEST_SPLIT_PATH}")
print(f"validation report   : {VALIDATION_REPORT_PATH}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: RESTRICTED-WINDOW FEATURE ENGINEERING FUNCTIONS (SAME AS NOTEBOOK 35)
# =============================================================================
_section("SECTION 4: Restricted-Window Feature Engineering Functions")

with open(RAW_TRAIN_DATA_PATH, "r", encoding="utf-8") as f:
    _train_header = f.readline().strip().split(",")
CATEGORICAL = ["B_30", "B_38", "D_63", "D_64", "D_66", "D_68",
               "D_114", "D_116", "D_117", "D_120", "D_126"]
feature_columns = [c for c in _train_header if c not in ("customer_ID", "S_2")]
categorical_cols = [c for c in feature_columns if c in CATEGORICAL]
numeric_cols = [c for c in feature_columns if c not in CATEGORICAL]
_top_corr_cols = list(VALIDATION_REPORT["top_5_correlated_with_target"].keys())


def build_early_window_store(csv_path: Path, num_cols: list, cat_cols: list, k: int) -> "pl.DataFrame":
    """Identical to Notebook 35's build_early_window_store() -- re-declared
    here (this platform's notebooks are not yet wired to a shared module, so
    reproducible-by-construction logic like this is duplicated verbatim
    rather than imported; see root ROADMAP.md)."""
    schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
    for c in cat_cols:
        schema_overrides[c] = pl.Utf8
    for c in num_cols:
        schema_overrides[c] = pl.Float32
    _inf_clean_exprs = [
        pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
        for c in num_cols
    ]
    lf = (
        pl.scan_csv(str(csv_path), schema_overrides=schema_overrides)
        .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
        .with_columns(_inf_clean_exprs)
        .sort(["customer_ID", "S_2"])
        .with_columns(pl.int_range(pl.len()).over("customer_ID").cast(pl.Float32).alias("_t_idx"))
        .filter(pl.col("_t_idx") < k)
    )
    agg_exprs = []
    for c in num_cols:
        agg_exprs += [
            pl.col(c).mean().alias(f"{c}_mean"), pl.col(c).std().alias(f"{c}_std"),
            pl.col(c).min().alias(f"{c}_min"), pl.col(c).max().alias(f"{c}_max"),
            pl.col(c).last().alias(f"{c}_last"), pl.col(c).first().alias(f"_first_{c}"),
            pl.cov(pl.col("_t_idx"), pl.col(c)).alias(f"_cov_{c}"),
            pl.when(pl.col(c).is_not_null()).then(pl.col("_t_idx")).otherwise(None).var().alias(f"_var_t_{c}"),
        ]
    for c in cat_cols:
        agg_exprs += [pl.col(c).last().alias(f"{c}_last"), pl.col(c).drop_nulls().n_unique().alias(f"{c}_nunique")]
    agg_exprs += [pl.len().alias("statement_count"),
                  (pl.col("S_2").max() - pl.col("S_2").min()).dt.total_days().alias("tenure_days")]
    grouped = lf.group_by("customer_ID", maintain_order=False).agg(agg_exprs)
    _trend_exprs = []
    for c in num_cols:
        _trend_exprs.append(
            pl.when((pl.col(f"_var_t_{c}").is_not_null()) & (pl.col(f"_var_t_{c}") > 0))
            .then(pl.col(f"_cov_{c}") / pl.col(f"_var_t_{c}")).otherwise(None).alias(f"{c}_trend_slope")
        )
        _trend_exprs.append((pl.col(f"{c}_last") - pl.col(f"_first_{c}")).alias(f"{c}_trend_delta"))
    _keep_cols = ["customer_ID", "statement_count", "tenure_days"]
    for c in num_cols:
        _keep_cols += [f"{c}_mean", f"{c}_std", f"{c}_min", f"{c}_max", f"{c}_last"]
    for c in cat_cols:
        _keep_cols += [f"{c}_last", f"{c}_nunique"]
    _keep_cols += [f"{c}_trend_slope" for c in num_cols] + [f"{c}_trend_delta" for c in num_cols]
    result = grouped.with_columns(_trend_exprs).select(_keep_cols).sort("customer_ID")
    return result.collect(engine="streaming")


def build_ratio_features(df: "pl.DataFrame", num_cols: list) -> "pl.DataFrame":
    exprs, new_col_names = [], []
    for c in num_cols:
        _mean, _std, _min, _max, _last = f"{c}_mean", f"{c}_std", f"{c}_min", f"{c}_max", f"{c}_last"
        if not all(col in df.columns for col in (_mean, _std, _min, _max, _last)):
            continue
        _ratio_name, _range_name, _cov_name = (
            f"{c}_ratio_last_to_mean", f"{c}_range", f"{c}_coeff_of_variation",
        )
        exprs.append(pl.when((pl.col(_mean).is_not_null()) & (pl.col(_mean) != 0))
                     .then(pl.col(_last) / pl.col(_mean)).otherwise(None).alias(_ratio_name))
        exprs.append((pl.col(_max) - pl.col(_min)).alias(_range_name))
        exprs.append(pl.when((pl.col(_mean).is_not_null()) & (pl.col(_mean) != 0))
                     .then(pl.col(_std) / pl.col(_mean)).otherwise(None).alias(_cov_name))
        new_col_names += [_ratio_name, _range_name, _cov_name]
    return df.with_columns(exprs).select(["customer_ID"] + new_col_names)


def build_interaction_features(df: "pl.DataFrame", top_corr_cols: list) -> "pl.DataFrame":
    _cols = [c for c in top_corr_cols if c in df.columns]
    exprs, pairs = [], []
    for i in range(len(_cols)):
        for j in range(i + 1, len(_cols)):
            c1, c2 = _cols[i], _cols[j]
            _name = f"interaction_{c1}_x_{c2}"
            exprs.append((pl.col(c1) * pl.col(c2)).alias(_name))
            pairs.append(_name)
    if not pairs:
        return df.select(["customer_ID"])
    return df.select(["customer_ID"] + _cols).with_columns(exprs).select(["customer_ID"] + pairs)


print("Feature-engineering functions re-declared (identical to Notebook 35).")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: REBUILD WINNING K'S FEATURE SET & RETRAIN THE FINAL MODEL
# =============================================================================
_section(f"SECTION 5: Rebuild K={WINNING_K}'s Feature Set & Retrain the Final Model")

# --- Notebook 35 evaluated every K but did not persist a fitted model object
#     for each one (only the holdout metrics). This retrains ONLY the winning
#     K, deterministically (same RANDOM_SEED, same data, same hyperparameters
#     Notebook 35 used) -- not a new decision, a real reproduction of the
#     exact model whose metrics are already reported above. ---
labels_df = pl.read_csv(str(RAW_TRAIN_LABELS_PATH), schema_overrides={"customer_ID": pl.Utf8, "target": pl.Int8})
train_ids_set = set(pl.read_csv(str(TRAIN_SPLIT_PATH), columns=["customer_ID"])["customer_ID"].to_list())
val_ids_set = set(pl.read_csv(str(TEST_SPLIT_PATH), columns=["customer_ID"])["customer_ID"].to_list())

_t0 = time.time()
_base = build_early_window_store(RAW_TRAIN_DATA_PATH, numeric_cols, categorical_cols, WINNING_K)
_ratios = build_ratio_features(_base, numeric_cols)
_interactions = build_interaction_features(_base, _top_corr_cols)
engineered = (
    _base.join(_ratios, on="customer_ID", how="left")
    .join(_interactions, on="customer_ID", how="left")
    .join(labels_df, on="customer_ID", how="inner")
)
if engineered.shape[0] != _base.shape[0]:
    raise RuntimeError(f"Row count changed after joins ({_base.shape[0]:,} -> {engineered.shape[0]:,}).")
print(f"Rebuilt {engineered.shape[0]:,} customers x {engineered.shape[1]} columns in {time.time() - _t0:.1f}s")
del _base, _ratios, _interactions
gc.collect()

train_df = engineered.filter(pl.col("customer_ID").is_in(train_ids_set))
holdout_df = engineered.filter(pl.col("customer_ID").is_in(val_ids_set))
holdout_customer_ids = holdout_df.get_column("customer_ID").to_list()
del engineered
gc.collect()

non_feature_cols = {"customer_ID", "target"}
categorical_encode_cols = [f"{c}_last" for c in CATEGORICAL if c in categorical_cols]
numeric_feature_cols = [c for c in train_df.columns if c not in non_feature_cols and c not in categorical_encode_cols]
all_feature_cols = numeric_feature_cols + categorical_encode_cols

_inf_clean_exprs = [
    pl.when(pl.col(c).is_infinite() | pl.col(c).is_nan()).then(None).otherwise(pl.col(c)).cast(pl.Float32).alias(c)
    for c in numeric_feature_cols
]
train_df = train_df.with_columns(_inf_clean_exprs)
holdout_df = holdout_df.with_columns(_inf_clean_exprs)

label_encoders = {}
for c in categorical_encode_cols:
    train_df = train_df.with_columns(pl.col(c).cast(pl.Utf8).fill_null("__missing__").alias(c))
    holdout_df = holdout_df.with_columns(pl.col(c).cast(pl.Utf8).fill_null("__missing__").alias(c))
    _cats = sorted(train_df.get_column(c).unique().to_list())
    _mapping = {cat: i for i, cat in enumerate(_cats)}
    train_df = train_df.with_columns(pl.col(c).replace_strict(_mapping, default=-1, return_dtype=pl.Int32).alias(c))
    holdout_df = holdout_df.with_columns(pl.col(c).replace_strict(_mapping, default=-1, return_dtype=pl.Int32).alias(c))
    label_encoders[c] = {"classes": _cats}

feature_medians = train_df.select(
    [pl.col(c).median().fill_null(0.0).alias(c) for c in numeric_feature_cols]
).to_dicts()[0]
_impute_exprs = [pl.col(c).fill_null(feature_medians[c]) for c in numeric_feature_cols]
train_df = train_df.with_columns(_impute_exprs)
holdout_df = holdout_df.with_columns(_impute_exprs)

X_train = train_df.select(all_feature_cols).to_numpy().astype(np.float32, copy=False)
y_train = train_df.get_column("target").to_numpy().astype(np.int64, copy=False)
X_holdout = holdout_df.select(all_feature_cols).to_numpy().astype(np.float32, copy=False)
y_holdout = holdout_df.get_column("target").to_numpy().astype(np.int64, copy=False)
del train_df, holdout_df
gc.collect()

final_model = XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    tree_method="hist", n_jobs=WARP_THREAD_COUNT, random_state=RANDOM_SEED,
    eval_metric="auc", verbosity=0,
)
final_model.fit(X_train, y_train)
proba_train = final_model.predict_proba(X_train)[:, 1]
proba_holdout = final_model.predict_proba(X_holdout)[:, 1]

_reproduced_auc = float(roc_auc_score(y_holdout, proba_holdout))
_reproduced_amex = float(amex_metric_numpy(y_holdout, proba_holdout))
print(f"Reproduced holdout AUC : {_reproduced_auc:.4f}  (Notebook 35 reported {WINNING_K_RESULT['holdout_auc']:.4f})")
print(f"Reproduced holdout AMEX: {_reproduced_amex:.4f}  (Notebook 35 reported {WINNING_K_RESULT['holdout_amex_metric']:.4f})")
_reproduction_matches = abs(_reproduced_auc - WINNING_K_RESULT["holdout_auc"]) < 1e-6
print(f"Reproduction matches Notebook 35 exactly (same seed, same data): {_reproduction_matches}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: BOOTSTRAP CONFIDENCE INTERVAL ON HOLDOUT AUC
# =============================================================================
_section("SECTION 6: Bootstrap Confidence Interval on Holdout AUC")

N_BOOTSTRAP = 2000
_rng = np.random.default_rng(RANDOM_SEED)
_n_holdout = len(y_holdout)
_boot_aucs = np.empty(N_BOOTSTRAP, dtype=np.float64)
for _b in range(N_BOOTSTRAP):
    _idx = _rng.integers(0, _n_holdout, size=_n_holdout)
    _yt, _yp = y_holdout[_idx], proba_holdout[_idx]
    if _yt.min() == _yt.max():
        _boot_aucs[_b] = np.nan  # degenerate resample (all-one-class); excluded below
    else:
        _boot_aucs[_b] = roc_auc_score(_yt, _yp)
_valid_boots = _boot_aucs[~np.isnan(_boot_aucs)]
AUC_CI_LOWER, AUC_CI_UPPER = np.percentile(_valid_boots, [2.5, 97.5])
print(f"Bootstrap resamples: {N_BOOTSTRAP:,} (valid: {len(_valid_boots):,}, random_state={RANDOM_SEED})")
print(f"Holdout AUC 95% CI: [{AUC_CI_LOWER:.4f}, {AUC_CI_UPPER:.4f}]  (point estimate {_reproduced_auc:.4f})")
_ci_excludes_random = AUC_CI_LOWER > 0.5
print(f"95% CI entirely above random (0.5) -- real, non-chance predictive power: {_ci_excludes_random}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: CALIBRATION CHECK -- PREDICTED PD DECILES VS. REAL OBSERVED DEFAULT RATE
# =============================================================================
_section("SECTION 7: Calibration Check")

_calib_df = pd.DataFrame({"proba": proba_holdout, "target": y_holdout})
_calib_df["decile"] = pd.qcut(_calib_df["proba"], q=10, labels=False, duplicates="drop")
calibration_table = (
    _calib_df.groupby("decile")
    .agg(n=("target", "size"), mean_predicted_pd=("proba", "mean"), observed_default_rate=("target", "mean"))
    .reset_index()
)
calibration_table["abs_gap"] = (calibration_table["mean_predicted_pd"] - calibration_table["observed_default_rate"]).abs()
print(calibration_table.round(4).to_string(index=False))
MEAN_CALIBRATION_GAP = float(calibration_table["abs_gap"].mean())
print(f"\nMean |predicted - observed| gap across deciles (real, measured): {MEAN_CALIBRATION_GAP:.4f}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: SPLIT-HALF POPULATION STABILITY (PSI) ON THE PREDICTED SCORE
# =============================================================================
_section("SECTION 8: Split-Half Population Stability (PSI) on the Predicted Score")

# --- Same honest framing this platform already documents for Notebook 21/
#     monitoring_job.py's PSI: a random split-half stability proxy on the
#     single available holdout population, not a genuine time-based drift
#     measurement -- there is no second real time period to compare against
#     yet. Bin edges come from the TRAIN score distribution's real deciles
#     (matching this platform's monitoring convention), applied to two
#     random halves of the holdout. ---
_train_edges = np.quantile(proba_train, np.linspace(0, 1, 11))
_train_edges[0], _train_edges[-1] = -np.inf, np.inf
_perm = _rng.permutation(_n_holdout)
_half = _n_holdout // 2
_half_a = proba_holdout[_perm[:_half]]
_half_b = proba_holdout[_perm[_half:]]
_share_a = np.histogram(_half_a, bins=_train_edges)[0] / len(_half_a)
_share_b = np.histogram(_half_b, bins=_train_edges)[0] / len(_half_b)
_share_a = np.clip(_share_a, 1e-4, None)
_share_b = np.clip(_share_b, 1e-4, None)
SCORE_PSI_SPLIT_HALF = float(((_share_a - _share_b) * np.log(_share_a / _share_b)).sum())
_psi_target = 0.10  # ASSUMPTION: standard industry PSI stability threshold (<0.10 = no significant shift)
print(f"Split-half PSI on predicted score: {SCORE_PSI_SPLIT_HALF:.4f}  (target < {_psi_target}, "
      f"{'PASS' if SCORE_PSI_SPLIT_HALF < _psi_target else 'FAIL'})")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: STATISTICAL VALIDATION SUMMARY TABLE
# =============================================================================
_section("SECTION 9: Statistical Validation Summary Table")

statistical_validation_rows = [
    {"test": "Holdout AUC (reproduced)", "value": round(_reproduced_auc, 4), "target": ">0.5", "pass": bool(_reproduced_auc > 0.5)},
    {"test": "Holdout AUC 95% CI lower bound", "value": round(float(AUC_CI_LOWER), 4), "target": ">0.5", "pass": bool(_ci_excludes_random)},
    {"test": "Mean calibration gap (deciles)", "value": round(MEAN_CALIBRATION_GAP, 4), "target": "<0.05", "pass": bool(MEAN_CALIBRATION_GAP < 0.05)},
    {"test": "Split-half score PSI", "value": round(SCORE_PSI_SPLIT_HALF, 4), "target": f"<{_psi_target}", "pass": bool(SCORE_PSI_SPLIT_HALF < _psi_target)},
    {"test": f"AUC retention vs. full history (K={WINNING_K})", "value": round(WINNING_K_RESULT["auc_retention_pct_of_full_history"], 1),
     "target": f">={KPI_TARGETS['min_auc_retention_vs_full_history']:.0%}", "pass": bool(MEETS_KPI)},
]
statistical_validation_df = pd.DataFrame(statistical_validation_rows)
statistical_validation_path = EPD_DEPLOYMENT_DIR / "early_default_statistical_validation.csv"
statistical_validation_df.to_csv(statistical_validation_path, index=False)
print(statistical_validation_df.to_string(index=False))
ALL_STAT_CHECKS_PASS = bool(statistical_validation_df["pass"].all())
print(f"\nAll statistical checks pass: {ALL_STAT_CHECKS_PASS}")
print(f"\u2705 Saved -> {statistical_validation_path}")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: HONEST LIMITATION -- DEPLOYMENT SCOPE & ASSUMPTIONS
# =============================================================================
_section("SECTION 10: Honest Limitation -- Deployment Scope & Assumptions")

DEPLOYMENT_LIMITATION = {
    "fixed_window_assumption": (
        f"This model was trained and validated on customers' first {WINNING_K} chronological statements "
        f"ONLY. It assumes exactly (or up to) {WINNING_K} statements are available at scoring time -- a "
        f"customer with MORE history available should be scored by Problem 1's full-history champion model "
        f"instead (it retains more signal), not this restricted model. This service does not enforce that "
        f"choice; it is an operational integration decision for whoever calls it."
    ),
    "kpi_status": (
        f"MEETS the {KPI_TARGETS['min_auc_retention_vs_full_history']:.0%} AUC-retention KPI set in Notebook 34."
        if MEETS_KPI else
        f"DOES NOT MEET the {KPI_TARGETS['min_auc_retention_vs_full_history']:.0%} AUC-retention KPI set in "
        f"Notebook 34 -- this is the best-performing candidate window tested, packaged here for completeness "
        f"and so validation/deployment tooling exists, but it is NOT RECOMMENDED FOR PRODUCTION USE until a "
        f"future run either finds a viable window or the KPI target itself is revisited."
    ),
    "data_limitation": (
        "Same limitation Notebook 34 documented: this dataset has no account-origination date, so 'early' "
        "means each customer's own earliest statements, not literal time-since-account-opening."
    ),
    "single_architecture_scope": (
        f"Only the champion architecture ({CHAMPION_NAME}) was evaluated (see Notebook 35's Section 7 scope "
        "note) -- a full model-zoo tournament at this window length was not run."
    ),
}
for _k, _v in DEPLOYMENT_LIMITATION.items():
    print(f"{_k}:\n  {_v}\n")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: PERSIST MODEL & PREPROCESSING ARTIFACTS
# =============================================================================
_section("SECTION 11: Persist Model & Preprocessing Artifacts")

MODEL_FILENAME = f"early_default_xgboost_k{WINNING_K}.joblib"
model_path = MODELS_SUBDIR / MODEL_FILENAME
joblib.dump(final_model, model_path)
print(f"\u2705 {model_path.name:<40} {model_path.stat().st_size / 1e6:>8,.2f} MB")

preprocessing_path = MODELS_SUBDIR / "preprocessing_artifacts.joblib"
joblib.dump({
    "label_encoders": label_encoders,
    "feature_medians": feature_medians,
    "all_feature_cols": all_feature_cols,
    "categorical_encode_cols": categorical_encode_cols,
    "numeric_feature_cols": numeric_feature_cols,
    "k": WINNING_K,
    "model_filename": MODEL_FILENAME,
}, preprocessing_path)
print(f"\u2705 {preprocessing_path.name:<40} {preprocessing_path.stat().st_size / 1e6:>8,.2f} MB")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: GENERATE early_default_service.py -- REAL, RUNNABLE FASTAPI SERVICE
# =============================================================================
_section("SECTION 12: Generate early_default_service.py -- Real FastAPI Service")

# --- Same generation pattern Notebook 10 (Problem 1) and Notebook 22
#     (Problem 2) established: plain string-list build (avoids f-string
#     brace-escaping on this generated source's own literal braces), same
#     AMEX_PROJECT_ROOT env-var convention, model+preprocessing paths baked
#     in as tokens so the generated file is genuinely standalone. ---
_default_project_root_str = str(PROJECT_ROOT)
_models_subdir_str = str(MODELS_SUBDIR)

EARLY_DEFAULT_SERVICE_TEMPLATE = "\n".join([
    "# AMEX Enterprise Credit Risk Platform -- Early Payment Default Scoring API.",
    "# Auto-generated by 36_early_payment_default_validation_deployment.ipynb.",
    f"# Scores a customer using ONLY their first {WINNING_K} chronological statements -- see /model-info.",
    "# Run with:",
    "#     uvicorn early_default_service:app --host 0.0.0.0 --port 8002",
    "import os",
    "from pathlib import Path",
    "from typing import Optional",
    "",
    "import joblib",
    "import numpy as np",
    "from fastapi import FastAPI, HTTPException",
    "from pydantic import BaseModel, create_model",
    "",
    "MODELS_DIR = Path(os.environ.get(\"AMEX_EPD_MODELS_DIR\", r\"__MODELS_DIR_TOKEN__\"))",
    "",
    "preprocessing_artifacts = joblib.load(MODELS_DIR / \"preprocessing_artifacts.joblib\")",
    "label_encoders = preprocessing_artifacts[\"label_encoders\"]",
    "feature_medians = preprocessing_artifacts[\"feature_medians\"]",
    "all_feature_cols = preprocessing_artifacts[\"all_feature_cols\"]",
    "categorical_encode_cols = preprocessing_artifacts[\"categorical_encode_cols\"]",
    "numeric_feature_cols = preprocessing_artifacts[\"numeric_feature_cols\"]",
    "EARLY_WINDOW_K = preprocessing_artifacts[\"k\"]",
    "model = joblib.load(MODELS_DIR / preprocessing_artifacts[\"model_filename\"])",
    "",
    "_schema_fields = {}",
    "for _c in numeric_feature_cols:",
    "    _schema_fields[_c] = (Optional[float], None)",
    "for _c in categorical_encode_cols:",
    "    _schema_fields[_c] = (Optional[str], None)",
    "CustomerFeatures = create_model(\"CustomerFeatures\", **_schema_fields)",
    "",
    "",
    "class EarlyDefaultResponse(BaseModel):",
    "    customer_id: Optional[str] = None",
    "    predicted_pd: float",
    "    early_window_k: int",
    f"    meets_kpi_target: bool = {MEETS_KPI}",
    "",
    "",
    "app = FastAPI(",
    "    title=\"AMEX Enterprise Credit Risk Platform -- Early Payment Default Scoring API\",",
    f"    description=\"Scores default risk using only a customer's first {WINNING_K} chronological \"",
    "                \"statements. See /model-info for the real validation metrics behind this model.\",",
    "    version=\"1.0.0\",",
    ")",
    "",
    "",
    "@app.get(\"/health\")",
    "def health():",
    "    return {\"status\": \"ok\", \"early_window_k\": EARLY_WINDOW_K}",
    "",
    "",
    "@app.get(\"/model-info\")",
    "def model_info():",
    "    return {",
    "        \"early_window_k\": EARLY_WINDOW_K,",
    f"        \"meets_kpi_target\": {MEETS_KPI},",
    f"        \"holdout_auc\": {WINNING_K_RESULT['holdout_auc']!r},",
    f"        \"holdout_amex_metric\": {WINNING_K_RESULT['holdout_amex_metric']!r},",
    f"        \"auc_retention_pct_of_full_history\": {WINNING_K_RESULT['auc_retention_pct_of_full_history']!r},",
    f"        \"recommended_for_production\": {MEETS_KPI},",
    "    }",
    "",
    "",
    "@app.post(\"/score\", response_model=EarlyDefaultResponse)",
    "def score(features: CustomerFeatures, customer_id: Optional[str] = None):",
    "    row = features.dict() if hasattr(features, \"dict\") else features.model_dump()",
    "    x = np.zeros((1, len(all_feature_cols)), dtype=np.float32)",
    "    for i, col in enumerate(all_feature_cols):",
    "        val = row.get(col)",
    "        if col in categorical_encode_cols:",
    "            classes = label_encoders[col][\"classes\"]",
    "            mapping = {cat: idx for idx, cat in enumerate(classes)}",
    "            x[0, i] = mapping.get(val if val is not None else \"__missing__\", -1)",
    "        else:",
    "            if val is None or (isinstance(val, float) and np.isnan(val)):",
    "                val = feature_medians[col]",
    "            x[0, i] = val",
    "    try:",
    "        pd_score = float(model.predict_proba(x)[:, 1][0])",
    "    except Exception as exc:",
    "        raise HTTPException(status_code=500, detail=\"Scoring failed: \" + str(exc))",
    "    return EarlyDefaultResponse(customer_id=customer_id, predicted_pd=pd_score, early_window_k=EARLY_WINDOW_K)",
    "",
])
EARLY_DEFAULT_SERVICE_SOURCE = EARLY_DEFAULT_SERVICE_TEMPLATE.replace("__MODELS_DIR_TOKEN__", _models_subdir_str)

service_py_path = API_SUBDIR / "early_default_service.py"
with open(service_py_path, "w", encoding="utf-8") as f:
    f.write(EARLY_DEFAULT_SERVICE_SOURCE)
compile(EARLY_DEFAULT_SERVICE_SOURCE, str(service_py_path), "exec")
print(f"Generated {len(EARLY_DEFAULT_SERVICE_SOURCE.splitlines())} lines, syntax-checked OK.")
print(f"\u2705 Saved -> {service_py_path}")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: GENERATE .env.example & requirements-api.txt
# =============================================================================
_section("SECTION 13: Generate .env.example & requirements-api.txt")

ENV_EXAMPLE = f"""# Copy to .env and edit if this machine's models folder differs from the default.
AMEX_EPD_MODELS_DIR={MODELS_SUBDIR}
"""
env_example_path = API_SUBDIR / ".env.example"
with open(env_example_path, "w", encoding="utf-8") as f:
    f.write(ENV_EXAMPLE)

_api_packages = ["fastapi", "uvicorn", "pydantic", "joblib", "numpy", "scikit-learn", "xgboost"]
_api_pkg_versions = {}
for _pkg in _api_packages:
    try:
        _api_pkg_versions[_pkg] = importlib_metadata.version(_pkg)
    except importlib_metadata.PackageNotFoundError:
        _api_pkg_versions[_pkg] = None

requirements_api_path = API_SUBDIR / "requirements-api.txt"
with open(requirements_api_path, "w", encoding="utf-8") as f:
    f.write(f"# Minimal runtime dependencies for early_default_service.py -- auto-generated "
             f"{datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
    for _pkg, _ver in _api_pkg_versions.items():
        f.write(f"{_pkg}=={_ver}\n" if _ver else f"# {_pkg}  -- not installed here\n")

print(f"\u2705 Saved -> {env_example_path}")
print(f"\u2705 Saved -> {requirements_api_path}")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: LIVE SELF-TEST -- IMPORT THE GENERATED SERVICE & DRIVE IT
# =============================================================================
_section("SECTION 14: Live Self-Test -- Import the Generated Service & Drive It")

# --- Imports the EXACT file just written to disk -- proves the delivered
#     artifact works, not just an in-notebook copy of the same logic. ---
os.environ["AMEX_EPD_MODELS_DIR"] = str(MODELS_SUBDIR)
_spec = importlib.util.spec_from_file_location("amex_early_default_service", str(service_py_path))
_service_module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_service_module)
client = TestClient(_service_module.app)

_health_resp = client.get("/health")
assert _health_resp.status_code == 200, f"/health returned {_health_resp.status_code}"
print(f"GET /health     -> {_health_resp.status_code}  {_health_resp.json()}")

_info_resp = client.get("/model-info")
assert _info_resp.status_code == 200, f"/model-info returned {_info_resp.status_code}"
print(f"GET /model-info -> {_info_resp.status_code}  {_info_resp.json()}")

_sample_idx = 0
SAMPLE_CUSTOMER_ID = holdout_customer_ids[_sample_idx]
# --- Categorical columns in X_holdout already hold the LABEL-ENCODED integer
#     (Section 5's preprocessing), not the original raw category string -- so
#     the payload must reverse-map each encoded integer back to its original
#     string via label_encoders[col]["classes"][encoded_index] (the generated
#     service re-encodes strings itself; sending it the already-encoded int
#     under a str-typed field would silently mismatch, which is exactly what
#     an earlier version of this self-test did before this fix). ---
SAMPLE_PAYLOAD = {}
for _i, _col in enumerate(all_feature_cols):
    _val = X_holdout[_sample_idx, _i]
    if _col in categorical_encode_cols:
        _classes = label_encoders[_col]["classes"]
        _encoded = int(_val)
        SAMPLE_PAYLOAD[_col] = _classes[_encoded] if 0 <= _encoded < len(_classes) else None
    else:
        SAMPLE_PAYLOAD[_col] = None if (isinstance(_val, float) and np.isnan(_val)) else float(_val)
EXPECTED_PD_DIRECT = float(proba_holdout[_sample_idx])

_score_resp = client.post("/score", params={"customer_id": SAMPLE_CUSTOMER_ID}, json=SAMPLE_PAYLOAD)
assert _score_resp.status_code == 200, f"/score returned {_score_resp.status_code}: {_score_resp.text}"
_api_result = _score_resp.json()
_api_pd = _api_result["predicted_pd"]
print(f"POST /score     -> {_score_resp.status_code}  {_api_result}")

_pd_diff = abs(_api_pd - EXPECTED_PD_DIRECT)
print(f"\nEnd-to-end check: API PD ({_api_pd:.6f}) vs. directly-computed PD ({EXPECTED_PD_DIRECT:.6f}) -- diff {_pd_diff:.8f}")

API_SELF_TEST_PASSED = _pd_diff < 1e-3
if API_SELF_TEST_PASSED:
    print("\n\u2705 MATCH -- the live API's preprocessing and scoring are verified consistent with direct computation.")
else:
    print("\n\u274c MISMATCH -- do not deploy early_default_service.py until this is resolved.")

if not API_SELF_TEST_PASSED:
    raise RuntimeError("Notebook 36's API self-test FAILED -- see \u274c line above. Not safe to proceed.")
print("\n\u2705 Section 14 complete.")


# =============================================================================
# SECTION 15: API LATENCY BENCHMARK
# =============================================================================
_section("SECTION 15: API Latency Benchmark")

N_API_LATENCY_SAMPLES = 150
_api_latencies_ms = []
for _ in range(N_API_LATENCY_SAMPLES):
    _t0 = time.perf_counter()
    _ = client.post("/score", json=SAMPLE_PAYLOAD)
    _api_latencies_ms.append((time.perf_counter() - _t0) * 1000.0)
_api_latencies_ms = np.array(_api_latencies_ms)
api_latency_summary = {
    "n_samples": N_API_LATENCY_SAMPLES,
    "p50_ms": round(float(np.percentile(_api_latencies_ms, 50)), 3),
    "p95_ms": round(float(np.percentile(_api_latencies_ms, 95)), 3),
    "p99_ms": round(float(np.percentile(_api_latencies_ms, 99)), 3),
    "max_ms": round(float(_api_latencies_ms.max()), 3),
}
print(f"/score latency over {N_API_LATENCY_SAMPLES} real TestClient calls: {api_latency_summary}")
print("\n\u2705 Section 15 complete.")


# =============================================================================
# SECTION 16: DEPLOYMENT READINESS CHECKLIST
# =============================================================================
_section("SECTION 16: Deployment Readiness Checklist")

deployment_readiness_rows = [
    {"dimension": "Model reproducibility", "status": "PASS" if _reproduction_matches else "FAIL"},
    {"dimension": "Statistical validation (all checks)", "status": "PASS" if ALL_STAT_CHECKS_PASS else "FAIL"},
    {"dimension": "AUC-retention KPI (Notebook 34 target)", "status": "MET" if MEETS_KPI else "NOT MET"},
    {"dimension": "Model + preprocessing artifacts persisted", "status": "PASS" if model_path.exists() and preprocessing_path.exists() else "FAIL"},
    {"dimension": "API self-test (live, generated service)", "status": "PASS" if API_SELF_TEST_PASSED else "FAIL"},
    {"dimension": "API p99 latency < 500ms", "status": "PASS" if api_latency_summary["p99_ms"] < 500 else "FAIL"},
    {"dimension": "Overall recommendation", "status": "RECOMMENDED FOR PRODUCTION" if MEETS_KPI and ALL_STAT_CHECKS_PASS else "NOT RECOMMENDED FOR PRODUCTION"},
]
deployment_readiness_df = pd.DataFrame(deployment_readiness_rows)
deployment_readiness_path = EPD_DEPLOYMENT_DIR / "deployment_readiness_checklist.csv"
deployment_readiness_df.to_csv(deployment_readiness_path, index=False)
print(deployment_readiness_df.to_string(index=False))
print(f"\u2705 Saved -> {deployment_readiness_path}")
print("\n\u2705 Section 16 complete.")


# =============================================================================
# SECTION 17: CHARTS
# =============================================================================
_section("SECTION 17: Charts")

CHARTS_DIR = EPD_DEPLOYMENT_DIR / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)


def _style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.3)


fig, ax = plt.subplots(figsize=(7, 4.5))
_ks_sorted = sorted(RESULTS_BY_K.keys())
_aucs = [RESULTS_BY_K[k]["holdout_auc"] for k in _ks_sorted]
ax.plot(_ks_sorted, _aucs, marker="o", linewidth=2, color="#2563eb", label="Restricted-window holdout AUC")
ax.axhline(FULL_HISTORY_AUC, color="#64748b", linestyle="--", label=f"Full-history AUC ({FULL_HISTORY_AUC:.4f})")
ax.axhline(FULL_HISTORY_AUC * KPI_TARGETS["min_auc_retention_vs_full_history"], color="#dc2626", linestyle=":",
           label=f"KPI floor ({KPI_TARGETS['min_auc_retention_vs_full_history']:.0%} retention)")
ax.scatter([WINNING_K], [WINNING_K_RESULT["holdout_auc"]], color="#16a34a", s=100, zorder=5, label=f"Selected: K={WINNING_K}")
ax.set_xlabel("Early window K (statements)")
ax.set_ylabel("Holdout AUC")
ax.set_title("AUC-Retention Curve -- Early Window vs. Full History")
ax.legend(fontsize=8)
_style_axes(ax)
chart1_path = CHARTS_DIR / "auc_retention_curve.png"
fig.tight_layout()
fig.savefig(chart1_path, dpi=150)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(calibration_table["mean_predicted_pd"], calibration_table["observed_default_rate"],
        marker="o", linewidth=2, color="#2563eb", label=f"K={WINNING_K} model (real deciles)")
_diag = [0, max(calibration_table["mean_predicted_pd"].max(), calibration_table["observed_default_rate"].max())]
ax.plot(_diag, _diag, color="#94a3b8", linestyle="--", label="Perfect calibration")
ax.set_xlabel("Mean predicted PD (decile)")
ax.set_ylabel("Observed default rate (decile)")
ax.set_title(f"Calibration -- K={WINNING_K} Restricted-Window Model")
ax.legend(fontsize=8)
_style_axes(ax)
chart2_path = CHARTS_DIR / "calibration_curve.png"
fig.tight_layout()
fig.savefig(chart2_path, dpi=150)
plt.close(fig)

print(f"\u2705 Saved -> {chart1_path}")
print(f"\u2705 Saved -> {chart2_path}")
print("\n\u2705 Section 17 complete.")


# =============================================================================
# SECTION 18: WORD REPORT
# =============================================================================
_section("SECTION 18: Word Report -- Early_Default_Validation_Deployment_Report.docx")

doc = Document()
doc.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
doc.add_paragraph("Phase 2, Problem 5: Early Payment Default Detection -- Validation & Deployment Report")
doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

doc.add_heading("1. Scope & Selected Window", level=1)
doc.add_paragraph(
    f"Statistical validation and deployment packaging for the early-window model at K={WINNING_K} "
    f"statements, selected from Notebook 35's real AUC-retention curve across candidates "
    f"{sorted(RESULTS_BY_K.keys())}. "
    + ("This window meets Notebook 34's AUC-retention KPI target." if MEETS_KPI else
       "IMPORTANT: none of the tested candidates met Notebook 34's AUC-retention KPI target on this real "
       "run -- this is the best-performing candidate, packaged for completeness, and is NOT RECOMMENDED "
       "FOR PRODUCTION until a future run finds a viable window.")
)

doc.add_heading("2. Statistical Validation Summary", level=1)
_t = doc.add_table(rows=1, cols=len(statistical_validation_df.columns))
_t.style = "Light Grid Accent 1"
for _i, _col in enumerate(statistical_validation_df.columns):
    _t.rows[0].cells[_i].text = _col.replace("_", " ").title()
for _, _row in statistical_validation_df.iterrows():
    _cells = _t.add_row().cells
    for _i, _col in enumerate(statistical_validation_df.columns):
        _cells[_i].text = str(_row[_col])

doc.add_heading("3. Honest Limitation -- Deployment Scope & Assumptions", level=1)
for _k, _v in DEPLOYMENT_LIMITATION.items():
    doc.add_paragraph(f"{_k.replace('_', ' ').title()}: {_v}")

doc.add_heading("4. Deployment Readiness Checklist", level=1)
_t3 = doc.add_table(rows=1, cols=len(deployment_readiness_df.columns))
_t3.style = "Light Grid Accent 1"
for _i, _col in enumerate(deployment_readiness_df.columns):
    _t3.rows[0].cells[_i].text = _col.replace("_", " ").title()
for _, _row in deployment_readiness_df.iterrows():
    _cells = _t3.add_row().cells
    for _i, _col in enumerate(deployment_readiness_df.columns):
        _cells[_i].text = str(_row[_col])

doc.add_heading("5. API Performance", level=1)
doc.add_paragraph(f"Latency over {api_latency_summary['n_samples']} real TestClient calls to /score: "
                   f"p50={api_latency_summary['p50_ms']}ms, p95={api_latency_summary['p95_ms']}ms, "
                   f"p99={api_latency_summary['p99_ms']}ms, max={api_latency_summary['max_ms']}ms.")

doc.add_heading("6. Charts", level=1)
for _cp, _cap in [(chart1_path, "AUC-retention curve across candidate early windows"),
                   (chart2_path, f"Calibration -- K={WINNING_K} model, real holdout deciles")]:
    doc.add_picture(str(_cp), width=Inches(6.0))
    _p = doc.add_paragraph(_cap)
    _p.alignment = WD_ALIGN_PARAGRAPH.CENTER

report_path = EPD_DEPLOYMENT_DIR / "Early_Default_Validation_Deployment_Report.docx"
doc.save(report_path)
print(f"\u2705 Saved -> {report_path}")
print("\n\u2705 Section 18 complete.")


# =============================================================================
# SECTION 19: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 19: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Model reproduction matches Notebook 35's reported holdout AUC", _reproduction_matches)
_all_checks_passed &= _check("Bootstrap CI is well-formed (lower <= point estimate <= upper)",
                              AUC_CI_LOWER <= _reproduced_auc <= AUC_CI_UPPER)
_all_checks_passed &= _check("Calibration table has 10 (or fewer, if ties) deciles", 1 <= len(calibration_table) <= 10)
_all_checks_passed &= _check("API self-test passed", API_SELF_TEST_PASSED)
_expected_files = [statistical_validation_path, deployment_readiness_path, model_path, preprocessing_path,
                    service_py_path, env_example_path, requirements_api_path, chart1_path, chart2_path, report_path]
for _fp in _expected_files:
    _all_checks_passed &= _check(f"{_fp.name} exists and is non-empty", _fp.exists() and _fp.stat().st_size > 0)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n\u2705 Section 19 complete -- all checks passed.")


# =============================================================================
# SECTION 20: WRITE NOTEBOOK 36 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 20: Write Notebook 36 Summary Artifact")

NB36_SUMMARY = {
    "notebook": "36_early_payment_default_validation_deployment.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "winning_k": WINNING_K,
    "meets_kpi_target": MEETS_KPI,
    "recommended_for_production": bool(MEETS_KPI and ALL_STAT_CHECKS_PASS),
    "reproduced_holdout_auc": _reproduced_auc,
    "reproduced_holdout_amex_metric": _reproduced_amex,
    "bootstrap_auc_ci": [float(AUC_CI_LOWER), float(AUC_CI_UPPER)],
    "mean_calibration_gap": MEAN_CALIBRATION_GAP,
    "split_half_score_psi": SCORE_PSI_SPLIT_HALF,
    "model_path": str(model_path),
    "preprocessing_path": str(preprocessing_path),
    "service_py_path": str(service_py_path),
    "report_path": str(report_path),
    "random_seed": RANDOM_SEED,
}
NB36_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_36_summary.json"
with open(NB36_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB36_SUMMARY, f, indent=2)
print(f"Wrote: {NB36_SUMMARY_PATH}")

_section("NOTEBOOK 36 COMPLETE")
print(f"Winning window        : K={WINNING_K}")
print(f"Meets KPI target       : {MEETS_KPI}")
print(f"Recommended for production: {NB36_SUMMARY['recommended_for_production']}")
print(f"Reproduced holdout AUC : {_reproduced_auc:.4f}  (95% CI [{AUC_CI_LOWER:.4f}, {AUC_CI_UPPER:.4f}])")
print(f"API self-test          : {'PASSED' if API_SELF_TEST_PASSED else 'FAILED'}")
print(f"Word report            : {report_path}")
print(
    "\nNext: 37_early_payment_default_reporting_packaging.ipynb -- financial-impact reporting and final "
    "packaging for Problem 5, closing out Phase 2."
)
